# IOAI — 2025 Contest Evading Ai Generated Text Detection (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, glob, zipfile
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/test.csv'):
    os.environ['KAGGLE_API_TOKEN'] = 'KGAT_5dd261fded8e0d7eb2c29d8d65dfabea'  # 내장 토큰(규칙 수락된 계정)
    os.system('pip -q install kaggle')
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi(); api.authenticate()
    api.competition_download_files('neoai-2025-evading-generated-text-detection', path='data', quiet=False)
    for zp in glob.glob('data/*.zip'):
        with zipfile.ZipFile(zp) as zf: zf.extractall('data')
        os.remove(zp)
os.system('pip -q install -U datasets')
print('데이터 준비:', 'data/test.csv' if os.path.exists('data/test.csv') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# NeoAI 2025 — Evading AI-Generated Text Detection

> **Northern Eurasia Olympiad in AI 2025 · Kaggle playground competition (적대적 NLP · LLM)**

**gemma-2-2b** 가 생성한 답변을 **가짜텍스트 탐지기가 "사람이 쓴 글"로 착각**하게 만드는 과제입니다.
`data/test.csv` 의 프롬프트 500개를 gemma-2-2b 로 생성해 `submission.csv`(`prompt,generation`)로 제출합니다.

**규칙**: 모델은 gemma-2-2b 고정, **프롬프트·샘플링 파라미터·생성 파이프라인은 고정** — 허용되는 유일한 레버는
**모델 내부 표현(활성/가중치) 수정(스티어링)** 입니다. test 로 학습 금지.

**채점**: 정답(탐지기 점수)은 캐글 숨김 채점에만 있어, **Submissions** 탭에서 `submission.csv` 를 **캐글 리더보드에
자동 제출**해 점수를 받습니다. GPU 필요.

⚠️ **아래 베이스라인 = vanilla gemma 생성**(스티어링 없음) → 탐지기가 AI 로 잘 잡아냄(캐글 ≈ **0.12**).
모델 내부 활성을 "사람다움" 방향으로 미는 **스티어링**으로 점수를 올립니다(모범답안 참고). 비-게이트 미러
`unsloth/gemma-2-2b`(동일 가중치) 사용 — HF 토큰 불필요.


In [ ]:
import os
os.environ.setdefault('TRITON_CACHE_DIR', '/tmp/triton_cache')   # 쓰기 가능 캐시
import torch, pandas as pd, time
from transformers import AutoModelForCausalLM, AutoTokenizer
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device', dev)

MID = 'unsloth/gemma-2-2b'   # 비-게이트 미러(동일 가중치)
tok = AutoTokenizer.from_pretrained(MID); tok.padding_side = 'left'
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MID, torch_dtype=torch.float16,
                                             attn_implementation='eager').to(dev).eval()
prompts = pd.read_csv('data/test.csv')['prompt'].tolist()
print('prompts', len(prompts))


In [ ]:
# BASELINE: vanilla 생성 (그리디, 128 토큰) — 스티어링 없음
def generate(plist):
    g = []
    for i in range(0, len(plist), 20):
        e = tok(plist[i:i+20], return_tensors='pt', padding=True).to(dev)
        with torch.no_grad():
            o = model.generate(**e, max_new_tokens=128, do_sample=False, cache_implementation='dynamic')
        g.extend(tok.batch_decode(o, skip_special_tokens=True))
    return g
t = time.time(); gens = generate(prompts); print('gen done', round(time.time()-t, 1), 's')
pd.DataFrame({'prompt': prompts, 'generation': gens}).to_csv('submission.csv', index=False)
print('wrote submission.csv', len(gens), '\n→ Submissions 탭에서 캐글 채점 (베이스라인 ≈ 0.12). 스티어링으로 올리세요')


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)